# OLS Return Prediction Model

This notebook implements an **Ordinary Least Squares (OLS)** model to predict **next-month stock excess returns (`y_next`)** using the **Fama-French five factors**.

The workflow mirrors the Random Forest model in the project:

1. Load the cleaned dataset.
2. Restrict predictions to **post-2022 data**.
3. Use a **36-month rolling training window**.
4. Estimate an OLS model each month.
5. Predict next-month returns.
6. Rank stocks by predicted return within each month.
7. Select the **Top 6 stocks per month** for portfolio construction.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# =========================
# Load cleaned dataset
# =========================
df_model = pd.read_csv("../data_process/data_clean_2.csv")

# Convert date column
df_model["MthCalDt"] = pd.to_datetime(df_model["MthCalDt"])

print("Original df_model shape:", df_model.shape)
print(
    "Original date range:", df_model["MthCalDt"].min(), "to", df_model["MthCalDt"].max()
)

display(df_model.head())

Original df_model shape: (1501178, 24)
Original date range: 1998-04-30 00:00:00 to 2025-11-28 00:00:00


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthCap,MthRet,ShrOut,MthPrc_Abs,DateKey,Mkt-RF,SMB,HML,RMW,CMA,RF,ExcessRet,y_next,ret_lag1,ret_lag2,ret_lag3,history_len,log_size
0,10001,36720410,EWST,7953,1998-04-30,8.8125,21158.81,0.007143,2401,8.8125,199804,0.0074,0.0001,0.0103,-0.0168,-0.0036,0.0043,0.002843,-0.018184,-0.012702,-0.010844,-0.004300,36,9.959812
1,10001,36720410,EWST,7953,1998-05-29,8.6875,20858.69,-0.014184,2401,8.6875,199805,-0.0305,-0.0300,0.0341,0.0103,0.0247,0.0040,-0.018184,0.001754,0.002843,-0.012702,-0.010844,37,9.945526
2,10001,36720410,EWST,7953,1998-06-30,8.6250,20725.88,0.005854,2403,8.6250,199806,0.0321,-0.0361,-0.0203,-0.0026,-0.0303,0.0041,0.001754,0.010493,-0.018184,0.002843,-0.012702,38,9.939138
3,10001,36720410,EWST,7953,1998-07-31,8.7500,21026.25,0.014493,2403,8.7500,199807,-0.0242,-0.0520,-0.0183,0.0164,0.0076,0.0040,0.010493,-0.004300,0.001754,-0.018184,0.002843,39,9.953527
4,10001,36720410,EWST,7953,1998-08-31,8.7500,21026.25,0.000000,2403,8.7500,199808,-0.1605,-0.0497,0.0357,0.0335,0.0578,0.0043,-0.004300,0.065686,0.010493,0.001754,-0.018184,40,9.953527


In [2]:
# =========================
# Use full cleaned dataset
# =========================
print("Full df shape:", df_model.shape)
print(
    "Full date range:",
    df_model["MthCalDt"].min(),
    "to",
    df_model["MthCalDt"].max(),
)

# Feature columns
feature_cols = [
    "Mkt-RF",
    "SMB",
    "HML",
    "RMW",
    "CMA",
    "ret_lag1",
    "ret_lag2",
    "ret_lag3",
    "log_size",
]

# Target column
target_col = "y_next"

# All available months
all_months = sorted(df_model["DateKey"].unique())

print("Number of months:", len(all_months))
print("First months:", all_months[:5])
print("Last months:", all_months[-5:])

Full df shape: (1501178, 24)
Full date range: 1998-04-30 00:00:00 to 2025-11-28 00:00:00
Number of months: 332
First months: [np.int64(199804), np.int64(199805), np.int64(199806), np.int64(199807), np.int64(199808)]
Last months: [np.int64(202507), np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511)]


In [3]:
# =========================
# Rolling 36-month OLS
# =========================

results = []

months = all_months  # use previously defined months

for i in range(36, len(months)):

    month = months[i]

    # previous 36 months
    train_months = months[i - 36 : i]

    train_data = df_model[df_model["DateKey"].isin(train_months)]
    test_data = df_model[df_model["DateKey"] == month]

    if len(train_data) == 0 or len(test_data) == 0:
        continue

    X_train = train_data[feature_cols]
    y_train = train_data[target_col]

    X_test = test_data[feature_cols]

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    temp = test_data.copy()
    temp["predicted_return"] = preds

    results.append(temp)

# combine all predictions
pred_df = pd.concat(results)

print("Prediction dataset shape:", pred_df.shape)
display(pred_df.head())

Prediction dataset shape: (1351094, 25)


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthCap,MthRet,ShrOut,MthPrc_Abs,DateKey,Mkt-RF,SMB,HML,RMW,CMA,RF,ExcessRet,y_next,ret_lag1,ret_lag2,ret_lag3,history_len,log_size,predicted_return
36,10001,36720410,EWST,7953,2001-04-30,9.75,24462.75,-0.025000,2509,9.75,200104,0.0794,-0.0088,-0.0464,-0.0318,-0.0319,0.0039,-0.028900,0.094236,0.034590,-0.016458,0.007421,72,10.104907,0.003146
268,10002,05978R10,SABC,7954,2001-04-30,11.38,97037.26,-0.241333,8527,11.38,200104,0.0794,-0.0088,-0.0464,-0.0318,-0.0319,0.0039,-0.245233,-0.036592,0.423094,0.058700,0.083035,72,11.482850,-0.016290
432,10016,81002230,SCTT,1728,2001-04-30,23.50,398654.00,0.062147,16964,23.50,200104,0.0794,-0.0088,-0.0464,-0.0318,-0.0319,0.0039,0.058247,-0.032136,-0.031673,-0.017350,0.025326,72,12.895849,0.001621
485,10025,00103110,AEPI,7975,2001-04-30,52.75,408337.75,0.170596,7741,52.75,200104,0.0794,-0.0088,-0.0464,-0.0318,-0.0319,0.0039,0.166696,0.114904,0.075141,0.105835,-0.163442,72,12.919850,-0.001956
710,10026,46603210,JJSF,7976,2001-04-30,20.60,175615.00,0.225279,8525,20.60,200104,0.0794,-0.0088,-0.0464,-0.0318,-0.0319,0.0039,0.221379,0.133208,0.038436,0.049261,-0.094619,72,12.076049,0.000613


In [4]:
# =========================
# Keep only post-2022 predictions
# =========================

pred_df_post2022 = pred_df[pred_df["MthCalDt"] >= "2022-01-01"].copy()

print(pred_df_post2022.shape)

(252411, 25)


In [5]:
# =========================
# Rank stocks each month
# =========================

pred_df_post2022["rank"] = pred_df_post2022.groupby("DateKey")["predicted_return"].rank(
    ascending=False
)

top6_ols = pred_df_post2022[pred_df_post2022["rank"] <= 6].copy()

print("Top-6 dataset shape:", top6_ols.shape)
display(top6_ols.head())

Top-6 dataset shape: (282, 26)


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthCap,MthRet,ShrOut,MthPrc_Abs,DateKey,Mkt-RF,SMB,HML,RMW,CMA,RF,ExcessRet,y_next,ret_lag1,ret_lag2,ret_lag3,history_len,log_size,predicted_return,rank
90775,12759,75989230,RENN,53760,2022-01-31,21.89,113083.74,0.491144,5166,21.89,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,0.442129,0.037003,-0.302113,0.037510,0.442129,36,11.635884,0.030169,3.0
114027,13323,29355A10,ENPH,54026,2022-01-31,140.47,18808090.18,-0.232153,133894,140.47,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.232153,0.186730,-0.268340,0.079308,0.442129,70,16.749798,0.029455,6.0
279590,18184,33813J10,FSR,56480,2022-01-31,11.81,1941162.46,-0.249205,164366,11.81,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.249205,0.033023,-0.264710,0.332710,0.095563,36,14.478798,0.029513,5.0
352868,25487,05377410,CAR,6391,2022-01-31,176.18,9866080.00,-0.150407,56000,176.18,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.150407,0.041208,-0.244901,0.442129,0.442129,312,16.104613,0.033653,1.0
1117072,87337,72919P20,PLUG,17303,2022-01-31,21.87,12633314.85,-0.225292,577655,21.87,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.225292,0.156379,-0.291693,0.041286,0.442129,94,16.351848,0.029649,4.0


In [6]:
# =========================
# Evaluate model accuracy (POST-2022 ONLY)
# =========================

mse = mean_squared_error(
    pred_df_post2022["y_next"], pred_df_post2022["predicted_return"]
)

mae = mean_absolute_error(
    pred_df_post2022["y_next"], pred_df_post2022["predicted_return"]
)

print("OLS Performance (Post-2022)")
print("MSE:", mse)
print("MAE:", mae)

OLS Performance (Post-2022)
MSE: 0.01014282360646887
MAE: 0.07024139289582162


In [7]:
pred_df_post2022["predicted_return"].describe()

count    252411.000000
mean          0.008118
std           0.016979
min          -0.082135
25%          -0.002645
50%           0.008633
75%           0.018519
max           0.086292
Name: predicted_return, dtype: float64

In [8]:
# =========================
# Save OLS selections
# =========================

top6_ols.to_csv("../data/ols_top6_by_month_post2022.csv", index=False)

print("OLS Top-6 selections saved.")

OLS Top-6 selections saved.
